# House Price Prediction Czech Republic

## 1. Setup & Data Loading

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score

df_raw = pd.read_csv('images_and_data/sreality_master.csv')
print(f'Dataset shape: {df_raw.shape}')
df_raw.head()

Dataset shape: (17340, 31)


,estate_id,title,locality,region,latitude,longitude,price_czk,price_per_m2,area_m2,category,...,vlastnictvi,energ_stitek,anuita,dist_vecerka_m,dist_obchod_m,dist_tram_m,dist_metro_m,dist_bus_mhd_m,url,scraped_at
0,1458614348,Prodej bytu 1+kk 36 m²,"Černá v Pošumaví, Český Krumlov",Jihočeský kraj,48.737864,14.102581,3290000.0,91389,36,1+kk,...,Osobní,C - Úsporná,NaN,723.0,NaN,NaN,NaN,481.0,https://www.sreality.cz/detail/prodej/byt/1+kk...,2026-07-18T21:41:29.040753+00:00
1,3749109836,Prodej bytu 1+kk 25 m²,"Litvínov - Horní Litvínov, Most",Ústecký kraj,50.610939,13.636989,899000.0,35960,25,1+kk,...,Družstevní,G - Mimořádně nehospodárná,NaN,28.0,1908.0,715.0,NaN,68.0,https://www.sreality.cz/detail/prodej/byt/1+kk...,2026-07-18T21:41:29.040753+00:00
2,569421900,Prodej bytu 1+kk 28 m²,"Černá v Pošumaví, Český Krumlov",Jihočeský kraj,48.737864,14.102581,2690000.0,96071,28,1+kk,...,Osobní,C - Úsporná,NaN,723.0,NaN,NaN,NaN,481.0,https://www.sreality.cz/detail/prodej/byt/1+kk...,2026-07-18T21:41:29.040753+00:00
3,4111769676,Prodej bytu 2+1 64 m²,"Ostrov, Karlovy Vary",Karlovarský kraj,50.307745,12.956295,3249000.0,50766,64,2+1,...,Osobní,C - Úsporná,NaN,231.0,1117.0,NaN,NaN,259.0,https://www.sreality.cz/detail/prodej/byt/2+1/...,2026-07-18T21:41:29.040753+00:00
4,3664875596,Prodej bytu 2+1 55 m²,"Chotěboř, Havlíčkův Brod",Kraj Vysočina,49.721356,15.671102,3390000.0,61636,55,2+1,...,Osobní,F - Velmi nehospodárná,NaN,36.0,1244.0,NaN,NaN,154.0,https://www.sreality.cz/detail/prodej/byt/2+1/...,2026-07-18T21:41:29.040753+00:00


## 2. Data Cleaning

In [5]:
# split
stavba_parts = df_raw['stavba'].str.split(',', expand=True)
loc_parts = df_raw['locality'].str.split(',', n=1, expand=True)

df_raw = df_raw.assign(
    # --- stavba ---
    construction = stavba_parts[0].str.strip(),
    condition    = stavba_parts[1].str.strip(),
    floor        = stavba_parts[2].str.extract(r'(\d+)')[0],

    # --- locality ---
    city     = loc_parts[0].str.split(' - ').str[0],
    district = loc_parts[0].str.split(' - ').str[1].fillna(loc_parts[1])
)

# Praha, use second part
df_raw.loc[df_raw['city'] == 'Praha', 'district'] = loc_parts[1]

# single-value locality, district = city
df_raw['district'] = df_raw['district'].fillna(df_raw['city'])

# drop columns
df_raw = df_raw.drop(columns=[
    'latitude','longitude','stavba','locality','estate_id','title','url',
    'scraped_at','premise','seller', 'price_per_m2'
])

print(f'Dataset shape: {df_raw.shape}')
df_raw.head()

Dataset shape: (17340, 25)


,region,price_czk,area_m2,category,is_new,has_video,has_3d,prislusenstvi,infrastruktura,topeni,...,dist_vecerka_m,dist_obchod_m,dist_tram_m,dist_metro_m,dist_bus_mhd_m,construction,condition,floor,city,district
0,Jihočeský kraj,3290000.0,36,1+kk,False,True,False,"Zařízeno, Sklep, Parkovací stání",Vodovod: Vodovod; Elektřina: 230V; Kanalizace:...,Zdroj vytápění: Tepelné čerpadlo; Otopná těles...,...,723.0,NaN,NaN,NaN,481.0,Cihlová,Novostavba,3,Černá v Pošumaví,Český Krumlov
1,Ústecký kraj,899000.0,25,1+kk,False,False,False,Výtah,Plyn: Plynovod; Kanalizace: Veřejná kanalizace...,Zdroj vytápění: Centrální dálkové,...,28.0,1908.0,715.0,NaN,68.0,Panelová,Velmi dobrý,10,Litvínov,Horní Litvínov
2,Jihočeský kraj,2690000.0,28,1+kk,False,True,False,"Zařízeno, Sklep, Parkovací stání",Vodovod: Vodovod; Kanalizace: Veřejná kanaliza...,Zdroj vytápění: Tepelné čerpadlo; Otopná těles...,...,723.0,NaN,NaN,NaN,481.0,Cihlová,Novostavba,3,Černá v Pošumaví,Český Krumlov
3,Karlovarský kraj,3249000.0,64,2+1,False,False,False,"Částečně zařízeno, Sklep o ploše 5 m²",Vodovod: Vodovod; Plyn: Plynovod; Elektřina: 2...,Zdroj vytápění: Centrální dálkové; Otopná těle...,...,231.0,1117.0,NaN,NaN,259.0,Panelová,Velmi dobrý,4,Ostrov,Karlovy Vary
4,Kraj Vysočina,3390000.0,55,2+1,False,False,False,"Zařízeno, Sklep o ploše 12 m², Parkovací stání",Vodovod: Vodovod; Elektřina: 230V; Kanalizace:...,NaN,...,36.0,1244.0,NaN,NaN,154.0,Smíšená,Velmi dobrý,1,Chotěboř,Havlíčkův Brod


In [6]:
print('=== Missing Values ===')
missing = df_raw.isnull().sum()
print(missing[missing > 0])

print('\n=== Zero-Price Records ===')
zero_price = (df_raw['price_czk'] == 0).sum()
print(f'{zero_price} records have price = 0 ({zero_price/len(df_raw)*100:.1f}% of data)')
print('These are likely data entry errors and will be removed.')


=== Missing Values ===
prislusenstvi      1141
infrastruktura     4726
topeni             6581
telekomunikace    11236
studna            17298
vlastnictvi          11
energ_stitek       2041
anuita            17029
dist_vecerka_m       41
dist_obchod_m      2974
dist_tram_m        9605
dist_metro_m      12708
dist_bus_mhd_m       11
construction         11
condition            11
floor                11
dtype: int64

=== Zero-Price Records ===
0 records have price = 0 (0.0% of data)
These are likely data entry errors and will be removed.


In [8]:
df_raw.info()
df_raw.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17340 entries, 0 to 17339
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   region          17340 non-null  object 
 1   price_czk       17340 non-null  float64
 2   area_m2         17340 non-null  int64  
 3   category        17340 non-null  object 
 4   is_new          17340 non-null  bool   
 5   has_video       17340 non-null  bool   
 6   has_3d          17340 non-null  bool   
 7   prislusenstvi   16199 non-null  object 
 8   infrastruktura  12614 non-null  object 
 9   topeni          10759 non-null  object 
 10  telekomunikace  6104 non-null   object 
 11  studna          42 non-null     object 
 12  vlastnictvi     17329 non-null  object 
 13  energ_stitek    15299 non-null  object 
 14  anuita          311 non-null    float64
 15  dist_vecerka_m  17299 non-null  float64
 16  dist_obchod_m   14366 non-null  float64
 17  dist_tram_m     7735 non-null  

,price_czk,area_m2,anuita,dist_vecerka_m,dist_obchod_m,dist_tram_m,dist_metro_m,dist_bus_mhd_m
count,1.734000e+04,17340.000000,3.110000e+02,17299.000000,14366.000000,7735.000000,4632.000000,17329.000000
mean,7.397256e+06,69.625894,2.089310e+06,419.671310,1136.093485,835.002069,1285.850389,191.470714
std,5.504094e+06,56.237487,3.088016e+06,510.866825,951.676713,1091.850507,1184.118607,167.243027
min,5.799000e+04,11.000000,0.000000e+00,0.000000,0.000000,4.000000,19.000000,1.000000
25%,3.990000e+06,50.000000,0.000000e+00,141.000000,504.000000,164.500000,414.000000,97.000000
50%,6.190000e+06,64.000000,2.302000e+03,264.000000,874.000000,341.000000,843.000000,156.000000
75%,8.999000e+06,81.000000,4.009033e+06,509.000000,1424.000000,1064.000000,1732.750000,243.000000
max,8.790000e+07,5989.000000,1.449100e+07,5902.000000,6382.000000,6402.000000,6211.000000,2659.000000
